In [6]:
# Core imports for the whole lab
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
print('Setup complete. pandas', pd.__version__)
raw = pd.DataFrame({
    'id':    [1, 2, 3, 4, 5, 6, 7, 7],
    'name':  ['Ana', 'Bo', 'Cy', 'Di', 'Eve', 'Fin', 'Gus', 'Gus'],
    'age':   [30, 25, np.nan, 41, -1, 38, 29, 29],
    'city':  [' Pune ', 'pune', 'DELHI', 'Delhi ', 'Mumbai', 'bombay', 'Pune.', 'Pune.'],
    'spend': ['120.5', '80.0', '200.2', 'N/A', '150.0', '99000', '110.0', '110.0'],
    'date':  ['2024-01-05', '2024-01-06', '2024-01-07', '2024-01-08',
              '2024-01-09', '2024-01-10', '2024-01-11', '2024-01-11'],
})


Setup complete. pandas 2.2.3


In [ ]:
#1A. A FEW COMMANDS REVEAL MOST PROBLEMS
df = raw.copy()
df.info()
#1B. MISSING COUNTS, DUPLICATES & RANGES
print('Missing per column:')
print(df.isna().sum())
print('\nDuplicate rows:', df.duplicated().sum())
print('\nNote: spend is type', df['spend'].dtype, "-> stored as text!")

In [4]:
print("excercise-1")
# 1. duplicate row count
print("Duplicate rows:", df.duplicated().sum())
# 2. missing per column
print("Missing per column:")
print(df.isna().sum())
# 3. Problems I can see:
# - There is a duplicate row.
# - The age column has a missing value.
# - The age column contains an invalid value (-1).
# - The spend column is stored as text instead of numeric data.
# - City names have inconsistent formatting/capitalization.
# - The date column is stored as text instead of datetime.


excercise-1
Duplicate rows: 1
Missing per column:
id       0
name     0
age      1
city     0
spend    0
date     0
dtype: int64


In [5]:
#2. Missing values — detect & handle
#2A. UNMASK DISGUISED MISSING VALUES
df['spend'] = pd.to_numeric(df['spend'], errors='coerce')
df['age']   = df['age'].replace(-1, np.nan)
print('Missing after unmasking:')
print(df[['age', 'spend']].isna().sum())
#  2B. HANDLE THE GAPS (impute)
df['age']   = df['age'].fillna(df['age'].median())
df['spend'] = df['spend'].fillna(df['spend'].median())
print('Missing after imputing:', df[['age', 'spend']].isna().sum().sum())

Missing after unmasking:
age      2
spend    1
dtype: int64
Missing after imputing: 0


In [7]:
print("excercise-2")
ex = raw.copy()
# 1. Unmask missing values
ex['spend'] = pd.to_numeric(ex['spend'], errors='coerce')
ex['age'] = ex['age'].replace(-1, np.nan)
# 2a. Dropna version
drop_version = ex.dropna()
# 2b. Median-impute version
impute_version = ex.copy()
impute_version['age'] = impute_version['age'].fillna(impute_version['age'].median())
impute_version['spend'] = impute_version['spend'].fillna(impute_version['spend'].median())
# 3. Compare row counts
print("Original rows:", len(ex))
print("Rows after dropna:", len(drop_version))
print("Rows after median imputation:", len(impute_version))
print("Rows lost by dropping:", len(ex) - len(drop_version))


excercise-2
Original rows: 8
Rows after dropna: 5
Rows after median imputation: 8
Rows lost by dropping: 3


In [8]:
#3. Duplicates & data types
#3A. DROP DUPLICATE ROWS
print('Before:', df.shape)
df = df.drop_duplicates()
print('After :', df.shape, '-> removed the repeated Gus row')
#3B. FIX DATA TYPES
df['date'] = pd.to_datetime(df['date'])
df['city'] = df['city'].astype('string')
print(df.dtypes)

Before: (8, 6)
After : (7, 6) -> removed the repeated Gus row
id                int64
name             object
age             float64
city     string[python]
spend           float64
date     datetime64[ns]
dtype: object


In [9]:
print("excercise-3")
ex = raw.copy()
# 1. Fix types: spend -> numeric, date -> datetime
ex['spend'] = pd.to_numeric(ex['spend'], errors='coerce')
ex['date'] = pd.to_datetime(ex['date'], errors='coerce')
# 2. Drop duplicate rows
ex = ex.drop_duplicates()
# 3. Print dtypes + shape
print("Data types:")
print(ex.dtypes)
print("\nFinal shape:")
print(ex.shape)


excercise-3
Data types:
id                int64
name             object
age             float64
city             object
spend           float64
date     datetime64[ns]
dtype: object

Final shape:
(7, 6)


In [10]:
#4.Outliers — detect with the IQR rule
#4A. THE IQR RULE
q1, q3 = df['spend'].quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f'Q1={q1:.1f}  Q3={q3:.1f}  IQR={iqr:.1f}')
print(f'Normal range: {low:.1f} to {high:.1f}')
outliers = df[(df['spend'] < low) | (df['spend'] > high)]
print('\nOutlier rows:')
print(outliers[['name', 'spend']])
#4B. ONE WAY TO TREAT THEM — CAP (winsorise)
df['spend_capped'] = df['spend'].clip(lower=low, upper=high)
print(df[['name', 'spend', 'spend_capped']])

Q1=115.2  Q3=175.1  IQR=59.8
Normal range: 25.5 to 264.9

Outlier rows:
  name    spend
5  Fin  99000.0
  name    spend  spend_capped
0  Ana    120.5       120.500
1   Bo     80.0        80.000
2   Cy    200.2       200.200
3   Di    120.5       120.500
4  Eve    150.0       150.000
5  Fin  99000.0       264.875
6  Gus    110.0       110.000


In [11]:
print("excercise-4")
# 1. Q1, Q3, IQR for 'age'
q1, q3 = df['age'].quantile([0.25, 0.75])
iqr = q3 - q1
print(f"Q1 = {q1:.1f}")
print(f"Q3 = {q3:.1f}")
print(f"IQR = {iqr:.1f}")
# 2. Lower & upper bounds
low = q1 - 1.5 * iqr
high = q3 + 1.5 * iqr
print(f"Lower bound = {low:.1f}")
print(f"Upper bound = {high:.1f}")
# 3. Rows outside the bounds
outliers = df[(df['age'] < low) | (df['age'] > high)]
print("\nAge outliers:")
print(outliers[['name', 'age']])


excercise-4
Q1 = 29.2
Q3 = 34.0
IQR = 4.8
Lower bound = 22.1
Upper bound = 41.1

Age outliers:
Empty DataFrame
Columns: [name, age]
Index: []


In [12]:
#5.Messy text & inconsistent categories
#5A. THE PROBLEM — ONE CITY, MANY SPELLINGS
print(df['city'].value_counts())   # ' Pune ', 'pune', 'Pune.' all look different!
#5B. STANDARDISE THE STRINGS
s = df['city'].astype('string')
s = s.str.strip()                       # trim whitespace
s = s.str.lower()                       # unify case
s = s.str.replace('.', '', regex=False) # drop stray punctuation
s = s.replace({'bombay': 'mumbai'})     # map known variants to one label
df['city'] = s
print(df['city'].value_counts())        # now clean categories

city
 Pune     1
pune      1
DELHI     1
Delhi     1
Mumbai    1
bombay    1
Pune.     1
Name: count, dtype: Int64
city
pune      3
delhi     2
mumbai    2
Name: count, dtype: Int64


In [16]:
print("exercise-5")
# Create the messy Series
messy = pd.Series(
    [' London ', 'london', 'LONDON', 'N.Y.', 'new york ', 'New York'],
    dtype='string'
)
# 1. strip + lower
messy = messy.str.strip().str.lower()
# 2. map 'n.y.' -> 'new york'
messy = messy.replace({'n.y.': 'new york'})
# 3. value_counts()
print(messy.value_counts())


exercise-5
london      3
new york    3
Name: count, dtype: Int64
